# 05 — Similarity

The Measures and their Calibration on the real Catalog (ADR-0013). `dayhike
build-similarity` turns every DayHike in Europe into a row of facts once and
stores them, with the Calibration, in `data/catalog.similarity/`. This notebook
reads that directory: it shows the facts, how far apart the Measures' raw
scales are and what Calibration does to them, checks every Measure on named
pairs and synthetic cases, and ranks a few Seeds against all of Europe under
different Measure weights.

**Pairs region:** the named verification pairs and the reversed-direction
check sit in lat 42.3–46.9, lon 13.3–19.5 (Slovenia + Croatia: Julian Alps,
karst, Velebit, the coast, the Pannonian lowland), where each name is a
unique `osm_name`. Ranking always scores against every DayHike in the Catalog.

It needs the local Catalog, `data/catalog.sqlite` plus
`data/catalog.geometry.sqlite`, with `sample-dem`, `sample-landcover`,
`sample-climate` and then `build-similarity` run. A missing file stops the
notebook with an error; it does not run on the committed OSM fixture.

A missing fact stays missing (NaN), never 0: a DayHike without elevation is
not a flat DayHike. A Measure that is missing for a pair drops out of that
pair's weighted mean.


In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dayhike import Catalog, geometry_store_path_for, rank_dayhikes, similarity_dir_for
from dayhike.similarity import (
    ALONG_KINDS,
    FACT_COLUMNS,
    MEASURE_NAMES,
    MEASURES,
    AmenityPoints,
    calibrate,
    measure_scores,
    profile_shape,
    score_rows,
    steepness,
    surface_class,
    walk_facts,
)


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)


repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    if (candidate / "pyproject.toml").is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("pyproject.toml: run the notebook inside the repository")

catalog_path = repo / "data" / "catalog.sqlite"
for path in (catalog_path, geometry_store_path_for(catalog_path), similarity_dir_for(catalog_path)):
    if not path.exists():
        raise FileNotFoundError(f"{path}: ingest, sample and build-similarity first (README)")

started = time.monotonic()
catalog = Catalog.open(catalog_path)
store = catalog.similarity
print(f"catalog_open_seconds\t{time.monotonic() - started:.1f}")
meta = json.loads((similarity_dir_for(catalog_path) / "meta.json").read_text())
print("similarity", similarity_dir_for(catalog_path))
print(f"dayhikes\t{len(store.ids)}")
print(f"facts_seconds\t{meta['facts_seconds']}\tcalibration_seconds\t{meta['calibration_seconds']}")
if np.isnan(store.column("temperature_c_01")).all():
    raise RuntimeError("the Catalog has no ClimateNormal: run sample-climate, then build-similarity")

BY_NAME = {measure.name: measure for measure in MEASURES}
names = np.array([hike.name for hike in catalog.dayhikes], dtype=object)
origins = np.array([hike.name_origin for hike in catalog.dayhikes], dtype=object)
trailhead = np.array([(hike.trailhead.latitude, hike.trailhead.longitude) for hike in catalog.dayhikes])
REGION_LAT = (42.3, 46.9)
REGION_LON = (13.3, 19.5)
in_region = (
    (trailhead[:, 0] >= REGION_LAT[0]) & (trailhead[:, 0] <= REGION_LAT[1])
    & (trailhead[:, 1] >= REGION_LON[0]) & (trailhead[:, 1] <= REGION_LON[1])
)
print(f"region_dayhikes\t{int(in_region.sum())}")
facts = pd.DataFrame(np.asarray(store.facts).T, columns=FACT_COLUMNS)
facts.insert(0, "name", names)


## Facts

One row per DayHike (`FACT_COLUMNS`, float32, yes/no as 0/1) plus its
ProfileShape (64 heights along the walk, scaled to 0–1). The CONTEXT.md terms
say what each one is; `dayhike.similarity` says how it is computed. The table
gives each fact's spread and missing share over all of Europe; the twelve
QueryMonth climate columns are left out of the table.


In [ ]:
shown = [name for name in FACT_COLUMNS if name[-3:-2] != "_"]
spread = facts[shown].describe(percentiles=[0.05, 0.5, 0.95]).T[["mean", "5%", "50%", "95%"]]
spread["missing"] = facts[shown].isna().mean()
print(spread.round(3).to_string())
print(f"profile_missing\t{np.isnan(np.asarray(store.profile)[:, 0]).mean():.3f}")

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, column in zip(axes.flat, [
    "length_m", "elevation_gain_m", "highest_m", "temperature_c",
    "dry_stretch_m", "tarmac_approach_m", "climb_grade", "southness",
]):
    values = facts[column].dropna()
    ax.hist(values.clip(upper=values.quantile(0.99)), bins=40, color="#0f766e")
    ax.set_title(column, fontsize=9)
fig.suptitle("Facts over every DayHike")
fig.tight_layout()
show_fig(fig)


## Measures and Calibration

Each Measure compares one fact of two DayHikes and gives 1 (alike) to 0
(unlike), or nothing when either lacks the fact:

| Kind | Rule | Measures |
| --- | --- | --- |
| `ratio` | min(a′, b′) / max(a′, b′), a′ = max(a, floor) | length, elevation_gain, peaks_around, precipitation, dry_stretch, climb_grade, steepest_km |
| `tolerance` | max(0, 1 − abs(a − b) / D) (water count capped at 3 first) | difficulty, highest, lowest, temperature, solar_radiation, southness, water_count, tarmac_approach |
| `mix` | Σ min(pᵢ, qᵢ) over fractions summing to 1 | landcover, surface |
| `agree` | 1 when the two yes/no flags agree (both absent agrees) | peak_summited, viewpoint_along, the five facilities flags, round_trip |
| `profile` | 1 − mean abs difference of the ProfileShapes, the better of forward or reversed | profile |

The raw scores live on very different scales. The table below scores 20,000
random pairs drawn here: the median random pair, the share of random
pairs at 0.6 or more, and the spread. A random pair of DayHikes is mostly a
dissimilar pair, yet on `profile` most of them score above 0.6.

Calibration maps each raw score to the share of random pairs that score lower
on that Measure, ties counting half (`quantiles.npy`, 1,024 sorted values per
Measure and per QueryMonth variant). After it, a random pair sits around 0.5
on every Measure and a weight of 100 against 50 means about twice the pull.


In [ ]:
left, right = np.random.default_rng(1).integers(0, len(store.ids), size=(2, 20_000))
left, right = left[left != right], right[left != right]
rows = []
for measure in MEASURES:
    raw = measure_scores(store, measure, left, right)
    raw = raw[~np.isnan(raw)]
    calibrated = calibrate(store, measure.name, raw)
    rows.append(
        {
            "measure": measure.name,
            "kind": measure.kind,
            "constant": measure.constant,
            "scored_pairs": len(raw) / len(left),
            "raw_median": np.median(raw),
            "raw_q05": np.quantile(raw, 0.05),
            "raw_ge_0.6": (raw >= 0.6).mean(),
            "raw_sd": raw.std(),
            "calibrated_median": np.median(calibrated),
            "calibrated_sd": calibrated.std(),
        }
    )
calibration = pd.DataFrame(rows).set_index("measure")
print(calibration.round(3).to_string())
for name, row in calibration.iterrows():
    print(
        f"calibration\t{name}\traw_median={row['raw_median']:.2f}\t"
        f"raw_ge_0.6={row['raw_ge_0.6']:.2f}\tcalibrated_median={row['calibrated_median']:.2f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, column, title in ((axes[0], "raw_median", "raw"), (axes[1], "calibrated_median", "calibrated")):
    ax.barh(calibration.index, calibration[column], color="#1d4ed8")
    ax.axvline(0.5, color="#c2410c", linewidth=1)
    ax.set_xlim(0, 1)
    ax.set_title(f"median random pair, {title}")
    ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
show_fig(fig)


## Named pairs

Pairs of real DayHikes picked by hand, by what a Hiker would say of the two
outings, before computing anything. Each group of Measures has a similar pair
and a dissimilar pair; some have more pairs. A pair's score is the equal-weight
mean of the group's calibrated Measures (the `raw` column is the same mean
before Calibration). A group PASSes when every similar pair scores above every
dissimilar pair (`margin` = lowest similar minus highest dissimilar, above 0).

`verify` lines are collected for the summary at the end.


In [ ]:
VERIFY = {}  # (subject, check) -> (score, PASS or FAIL), read by the summary at the end


def verify(subject, check, score, ok):
    result = "PASS" if ok else "FAIL"
    VERIFY[(subject, check)] = (score, result)
    print(f"verify\t{subject}\t{check}\t{score:.3f}\t{result}")


region_named = in_region & (origins == "osm_name")


def row_of(name):
    found = np.flatnonzero(region_named & (names == name))
    assert len(found) == 1, f"pair name must be a unique region osm_name: {name}"
    return int(found[0])


def pair_score(a, b, measures, calibrated=True):
    weights = dict.fromkeys(measures, 1.0)
    similarity, raw = score_rows(store, [row_of(a)], np.array([row_of(b)]), weights)
    return float(similarity[0] if calibrated else raw[0])


def verify_group(group, measures, pairs):
    table = pd.DataFrame(
        {
            "expected": expected,
            "a": a,
            "b": b,
            "raw": pair_score(a, b, measures, calibrated=False),
            "score": pair_score(a, b, measures),
        }
        for expected, a, b in pairs
    )
    print(table.round(3).to_string(index=False))
    for column in ("raw", "score"):
        by_group = table.groupby("expected")[column]
        margin = by_group.min()["similar"] - by_group.max()["dissimilar"]
        if column == "raw":
            print(f"raw_margin\t{group}\t{margin:.3f}")
        else:
            verify(group, "pairs_margin", margin, margin > 0)


GROUPS = {
    "effort": (["length", "elevation_gain", "difficulty"], [
        ("similar", "Zagorje - Završje - Kum", "Mosor-Kosa"),
        ("dissimilar", "Pot kralja Triglava", "Log - Mangart"),
    ]),
    "altitude": (["highest", "lowest"], [
        ("similar", "Vršič - Razor", "Kotovo sedlo - Mangart"),
        ("dissimilar", "Omiš - Kula", "Dolič - Triglav"),
    ]),
    "profile": (["profile"], [
        ("similar", "Koča pri Savici - Dom na Komni", "Zavala - Sv. Ante"),
        ("similar", "Učna pot škrata Bisera", "Maslinovi puti"),
        ("similar", "Šturmovci - Ptujsko jezero", "Iter Aquileiense - variante della Roggia"),
        ("dissimilar", "Koča pri Savici - Dom na Komni", "Nacionalni park Sjeverni Velebit II - Rožanski kukovi"),
        ("dissimilar", "Maslinovi puti", "Blejski razgledni trail"),
        ("dissimilar", "Šturmovci - Ptujsko jezero", "Zavala - Sv. Ante"),
    ]),
    "landscape": (["landcover", "peak_summited", "viewpoint_along", "peaks_around"], [
        ("similar", "Begunje - Cemarovnik - Velika Špička", "Rakitna - Vrbica"),
        ("similar", "Predmeja - Iztokova koča", "Mošćenice-Kremenjak/Brgud"),
        ("similar", "Dolič - Triglav", "Špremova pot"),
        ("dissimilar", "Planinarska staza 16", "Gorjanska pot"),
        ("dissimilar", "Predmeja - Iztokova koča", "Dolič - Triglav"),
        ("dissimilar", "Predmeja - Iztokova koča", "Zahum Idovac"),
    ]),
    "weather": (["temperature", "precipitation", "solar_radiation", "southness"], [
        ("similar", "Kopiščarjeva pot", "Frischaufova pot"),
        ("similar", "Zavala - Sv. Ante", "Baška Voda-magistrala(Novaci)"),
        ("similar", "Kopiščarjeva pot", "Koča v Krnici - Križ"),
        ("dissimilar", "Kopiščarjeva pot", "Zavala - Sv. Ante"),
        ("dissimilar", "Kopiščarjeva pot", "Slovenska pot"),
        ("dissimilar", "Koča v Grohotu - Raduha", "Planinarska staza Medvednica 3"),
    ]),
    "water": (["water_count", "dry_stretch"], [
        ("similar", "Stara Fužina - Dom v Vojah", "Kanjon Velika Paklenica"),
        ("similar", "Strogi rezervat prirode Bijele i Samarske stijene", "Guge-Brezovac"),
        ("similar", "Čez Ljubljanski grad", "Sentiero Tabor / Pešpot Tabor"),
        ("dissimilar", "Kanjon Velika Paklenica", "Strogi rezervat prirode Bijele i Samarske stijene"),
        ("dissimilar", "Sentiero natura / Pešpot za Miramar", "Wanderweg 338"),
        ("dissimilar", "Lendweg", "Šegova pot"),
    ]),
    "facilities": ([
        "hut_along", "restaurant_cafe_along", "shelter_along",
        "restaurant_cafe_trailhead", "parking_trailhead",
    ], [
        ("similar", "Cojzova koča - Skuta", "Krekova koča - Altemaver - Kremant"),
        ("similar", "Okrešelj - Kamniško sedlo", "Hudičev boršt - Kališče"),
        ("similar", "Kulinarische Weinberg-Wanderung", "Wanderweg 343"),
        ("dissimilar", "Šmartno - Koča pod Kremžarjevim vrhom", "Draževnik - Gradišče"),
        ("dissimilar", "Okrešelj - Kamniško sedlo", "Kulinarische Weinberg-Wanderung"),
        ("dissimilar", "Okrešelj - Kamniško sedlo", "Mali Alan - Dušice"),
    ]),
    "trail": (["surface", "tarmac_approach"], [
        ("similar", "Dom Paklenica - Jurline", "Stap - Jelova Ruja"),
        ("similar", "Planinarska staza Medvednica 18", "Planinarska staza Medvednica 36"),
        ("similar", "Jelovec - Žavcarjev vrh", "Bresternica - Žavcarjev vrh"),
        ("dissimilar", "Krč - Lišćani bunar", "314 Cres – Lungomare"),
        ("dissimilar", "Dom Paklenica - Jurline", "135 ~ Pot kurirjev in vezistov NOV Slovenije"),
        ("dissimilar", "Planinarska staza Medvednica 18", "Stepinčev put 2. etapa"),
    ]),
    "round_trip": (["round_trip"], [
        ("similar", "Po stopinjah pastirjev", "Anello della Pecorara"),
        ("similar", "Razgledni vrhovi nad Črnim Vrhom", "Verborgene Gletscherwelt Runde"),
        ("similar", "Arriacher Hofwanderweg-Variante", "Velika planina - Planina Dol"),
        ("dissimilar", "Po stopinjah pastirjev", "Velika planina - Planina Dol"),
        ("dissimilar", "Arriacher Hofwanderweg-Variante", "Verborgene Gletscherwelt Runde"),
    ]),
    "steepness": (["climb_grade", "steepest_km"], [
        ("similar", "Kalanova pot", "Uniški batovi"),
        ("dissimilar", "Uniški batovi", "Hüttenwanderweg"),
    ]),
}
for group, (measures, pairs) in GROUPS.items():
    print(f"\n{group}: {', '.join(measures)}")
    verify_group(group, measures, pairs)


### The profile scale, raw and calibrated

The three dissimilar profile pairs above score high raw: two ProfileShapes
that both go up and down share most of the unit square. Calibrated, they sit
near the bottom of the random pairs, while the similar pairs stay near the
top.


In [ ]:
fraction = np.linspace(0.0, 1.0, 64)
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharey=True)
for ax, (expected, a, b) in zip(axes.flat, GROUPS["profile"][1]):
    raw = pair_score(a, b, ["profile"], calibrated=False)
    calibrated_score = pair_score(a, b, ["profile"])
    print(f"profile_pair\t{expected}\t{a}\t{b}\traw={raw:.3f}\tcalibrated={calibrated_score:.3f}")
    for name, color in ((a, "#0f766e"), (b, "#c2410c")):
        ax.plot(fraction, np.asarray(store.profile)[row_of(name)], color=color, label=name)
    ax.set_title(f"{expected}: raw {raw:.2f}, calibrated {calibrated_score:.2f}", fontsize=9)
    ax.set_xlabel("fraction of length")
    ax.legend(fontsize=6)
fig.suptitle("Profile pairs: ProfileShape in stored direction")
fig.tight_layout()
show_fig(fig)


## Synthetic checks

For every Measure, over all of Europe:

- identity: the lowest raw score of a DayHike against itself, which must be 1.
- symmetry: the largest gap between A vs B and B vs A over 20,000 random
  pairs, which must be 0 up to float32 rounding (under 1e-6).

For the facts from DayHikeGeometry, on a small Julian Alps box (Trailhead lat
46.3–46.5, lon 13.6–14.0):

- reversed: the facts rebuilt from each DayHike's vertices in reverse order
  (`walk_facts`, `profile_shape`; the walked ways keep their walk order from
  the Trailhead) must give the stored facts again: the largest relative difference must be under 1e-4 (float32), and
  the profile Measure of a DayHike against its reversed self must be 1.


In [ ]:
rng = np.random.default_rng(0)
left, right = rng.integers(0, len(store.ids), size=(2, 20_000))
everyone = np.arange(len(store.ids))
for measure in MEASURES:
    own = measure_scores(store, measure, everyone, everyone)
    worst = float(np.nanmin(own))
    verify(measure.name, "identity", worst, abs(worst - 1.0) < 1e-6)
    gap = float(np.nanmax(np.abs(
        measure_scores(store, measure, left, right) - measure_scores(store, measure, right, left)
    )))
    verify(measure.name, "symmetry_max_gap", gap, gap < 1e-6)


In [ ]:
import sqlite3

CHECK_LAT = (46.3, 46.5)
CHECK_LON = (13.6, 14.0)
check_rows = np.flatnonzero(
    (trailhead[:, 0] >= CHECK_LAT[0]) & (trailhead[:, 0] <= CHECK_LAT[1])
    & (trailhead[:, 1] >= CHECK_LON[0]) & (trailhead[:, 1] <= CHECK_LON[1])
)
check_ids = [int(store.ids[row]) for row in check_rows]
print(f"check_dayhikes\t{len(check_ids)}")
geo = sqlite3.connect(f"file:{geometry_store_path_for(catalog_path)}?mode=ro", uri=True)
marks = ",".join("?" * len(check_ids))
vertex_rows = geo.execute(
    f"SELECT dayhike_id, latitude, longitude, ele FROM vertices WHERE dayhike_id IN ({marks}) "
    "ORDER BY dayhike_id, seq",
    check_ids,
).fetchall()
way_rows = geo.execute(
    f"SELECT dayhike_id, walked_m, tags FROM ways WHERE dayhike_id IN ({marks}) "
    "AND walk_order IS NOT NULL ORDER BY dayhike_id, walk_order",
    check_ids,
).fetchall()
lat_all = [lat for _, lat, _, _ in vertex_rows]
lon_all = [lon for _, _, lon, _ in vertex_rows]
pad = 0.05  # degrees: more than the 2 km Peaks-around radius
amenity_rows = geo.execute(
    "SELECT kind, latitude, longitude FROM amenities WHERE latitude BETWEEN ? AND ? "
    "AND longitude BETWEEN ? AND ?",
    (min(lat_all) - pad, max(lat_all) + pad, min(lon_all) - pad, max(lon_all) + pad),
).fetchall()
geo.close()
kinds = np.array([kind for kind, _, _ in amenity_rows], dtype=object)
points = np.array([(lat, lon) for _, lat, lon in amenity_rows], dtype=float)
along = AmenityPoints(points[np.isin(kinds, ALONG_KINDS)], kinds[np.isin(kinds, ALONG_KINDS)])
peaks = AmenityPoints(points[kinds == "peak"], kinds[kinds == "peak"])

chains, walked = {}, {}
for dayhike_id, lat, lon, ele in vertex_rows:
    chains.setdefault(dayhike_id, []).append((lat, lon, np.nan if ele is None else ele))
for dayhike_id, walked_m, tags in way_rows:
    walked.setdefault(dayhike_id, []).append((walked_m, surface_class(json.loads(tags))))

worst_gap, worst_profile = 0.0, 1.0
for row, dayhike_id in zip(check_rows, check_ids):
    chain = np.array(chains[dayhike_id])
    stored = {name: float(store.facts[FACT_COLUMNS.index(name), row]) for name in FACT_COLUMNS}
    for lat, lon, ele, ways in (
        (chain[:, 0], chain[:, 1], chain[:, 2], walked.get(dayhike_id, [])),
        (chain[::-1, 0], chain[::-1, 1], chain[::-1, 2], walked.get(dayhike_id, [])),
    ):
        rebuilt = walk_facts(lat, lon, ele, ways, along, peaks)
        for name, value in rebuilt.items():
            if np.isnan(value) and np.isnan(stored[name]):
                continue
            gap = abs(value - stored[name]) / max(abs(value), 1.0)
            worst_gap = max(worst_gap, gap)
    backward = profile_shape(chain[::-1, 0], chain[::-1, 1], chain[::-1, 2])
    forward = np.asarray(store.profile)[row]
    if backward is not None:
        score = 1.0 - min(np.abs(forward - backward).mean(), np.abs(forward - backward[::-1]).mean())
        worst_profile = min(worst_profile, score)
verify("geometry_facts", "reversed_max_relative_gap", worst_gap, worst_gap < 1e-4)
verify("profile", "reversed", worst_profile, abs(worst_profile - 1.0) < 1e-6)


## Ranking: one Seed, two weight sets

Every DayHike in the Catalog is scored against the Seed "Aljažev dom -
Kredarica - Triglav": a 5.8 km, 2,230 m climb from 1,000 m past the Kredarica
hut to Triglav at 2,800 m, with a Hut and a Shelter along the way and parking
at the Trailhead, and no restaurant or cafe.

| Set | Weights (Measures not named are excluded) |
| --- | --- |
| `facilities_heavy` | the five facilities flags 10 each; highest, lowest 15 each; landcover, peak_summited, viewpoint_along, peaks_around 5 each |
| `altitude_heavy` | highest, lowest 30 each; length, elevation_gain 7 each, difficulty 6; landcover, peak_summited, viewpoint_along, peaks_around 5 each |

`facilities_heavy` is "I care about huts and how high it goes, not about the
trees"; `altitude_heavy` matches height and work whatever the huts. `score` is
the similarity, the weighted mean of calibrated Measures, after Twins are
dropped (ADR-0016).


In [ ]:
LANDSCAPE = {"landcover": 5, "peak_summited": 5, "viewpoint_along": 5, "peaks_around": 5}
WEIGHT_SETS = {
    "facilities_heavy": {
        "hut_along": 10, "restaurant_cafe_along": 10, "shelter_along": 10,
        "restaurant_cafe_trailhead": 10, "parking_trailhead": 10,
        "highest": 15, "lowest": 15, **LANDSCAPE,
    },
    "altitude_heavy": {
        "highest": 30, "lowest": 30, "length": 7, "elevation_gain": 7, "difficulty": 6,
        **LANDSCAPE,
    },
}
SEED = "Aljažev dom - Kredarica - Triglav"
SHOW = ["length_m", "elevation_gain_m", "highest_m", "lowest_m", "hut_along",
        "shelter_along", "parking_trailhead", "round_trip", "climb_grade"]


def ranked(seeds, weights, top=5):
    """rank_dayhikes, its time, and the top rows with their facts."""
    started = time.monotonic()
    ranking = rank_dayhikes(catalog, seeds, weights)
    seconds = time.monotonic() - started
    seed_rows = [row_of(seed) for seed in seeds]
    head = ranking.items[:top]
    rows = np.searchsorted(store.ids, [item.dayhike.id for item in head])
    similarity, _ = score_rows(store, seed_rows, rows, weights)
    table = facts.loc[rows, ["name", *SHOW]].assign(
        score=[item.score for item in head], similarity=similarity
    )
    return ranking, table, seconds


print(facts.loc[[row_of(SEED)], ["name", *SHOW]].round(2).T.to_string())
tops = {}
for set_name, weights in WEIGHT_SETS.items():
    ranking, table, seconds = ranked([SEED], weights)
    tops[set_name] = table
    print(f"\n{set_name}: " + ", ".join(f"{name} {w}" for name, w in weights.items()))
    print(table.round(3).to_string(index=False))
    print(f"query_seconds\t{set_name}\t{seconds:.2f}\tcandidates\t{len(ranking.items)}")
    for _, row in table.iterrows():
        print(f"rank\t{set_name}\t{row['name']}\t{row['score']:.4f}\t{row['similarity']:.3f}")
shared = set(tops["facilities_heavy"]["name"]) & set(tops["altitude_heavy"]["name"])
print(f"top5_shared\t{len(shared)}")


### Weight on RoundTrip and on Steepness

A Seed that is a RoundTrip, "Po stopinjah pastirjev", ranked under
`altitude_heavy` as it is and with `round_trip` weighted 50 on top: with the
weight, the top five also close where they start. A steep Seed, "Uniški
batovi" (7.1 km at about 20 %), under `altitude_heavy` and with `climb_grade`
and `steepest_km` weighted 25 each on top: with the weight, the top five also
climb as steeply.


In [ ]:
ROUND_TRIP_SEED = "Po stopinjah pastirjev"
for set_name, weights in (
    ("altitude_heavy", WEIGHT_SETS["altitude_heavy"]),
    ("altitude_heavy_round_trip", {**WEIGHT_SETS["altitude_heavy"], "round_trip": 50}),
):
    _, table, _ = ranked([ROUND_TRIP_SEED], weights)
    same = int((table["round_trip"] == facts.at[row_of(ROUND_TRIP_SEED), "round_trip"]).sum())
    print(f"\n{set_name}")
    print(table.round(3).to_string(index=False))
    print(f"round_trip_top5_same\t{set_name}\t{same}\tof\t5")

STEEP_SEED = "Uniški batovi"
seed_grade = facts.at[row_of(STEEP_SEED), "climb_grade"]
print(f"\nseed\t{STEEP_SEED}\tclimb_grade\t{seed_grade:.3f}")
for set_name, weights in (
    ("altitude_heavy", WEIGHT_SETS["altitude_heavy"]),
    ("altitude_heavy_steepness", {**WEIGHT_SETS["altitude_heavy"], "climb_grade": 25, "steepest_km": 25}),
):
    _, table, _ = ranked([STEEP_SEED], weights)
    print(f"\n{set_name}")
    print(table.round(3).to_string(index=False))
    gap = (table["climb_grade"] - seed_grade).abs().median()
    print(f"steepness_top5_grade_gap\t{set_name}\t{gap:.3f}")


## Several Seeds: the maximum rule

With several Seeds, a candidate's similarity is its best similarity to any one
Seed, never the mean, and its `score` is that similarity. Two dissimilar Seeds: the alpine "Aljažev dom - Kredarica
- Triglav" and the Hvar coast climb "Zavala - Sv. Ante", under
`altitude_heavy`. Each `multi_seed` line prints the similarity and the two
per-Seed similarities; the cell asserts the maximum for every candidate.


In [ ]:
SEEDS = ["Aljažev dom - Kredarica - Triglav", "Zavala - Sv. Ante"]
weights = WEIGHT_SETS["altitude_heavy"]
seed_rows = [row_of(name) for name in SEEDS]
candidates = np.setdiff1d(everyone, seed_rows)
per_seed = [score_rows(store, [row], candidates, weights)[0] for row in seed_rows]
best, _ = score_rows(store, seed_rows, candidates, weights)
assert np.allclose(best, np.fmax(*per_seed), equal_nan=True), "similarity must be the higher of the two"
ranking, table, seconds = ranked(SEEDS, weights, top=10)
print(table.round(3).to_string(index=False))
print(f"query_seconds\tmulti_seed\t{seconds:.2f}")
for item in ranking.items[:10]:
    index = np.searchsorted(candidates, np.searchsorted(store.ids, item.dayhike.id))
    print(f"multi_seed\t{item.dayhike.name}\t{best[index]:.3f}\t{per_seed[0][index]:.3f}\t{per_seed[1][index]:.3f}")
won_by_first = per_seed[0] >= per_seed[1]
print(f"multi_seed_won\t{SEEDS[0]}\t{int(won_by_first.sum())}")
print(f"multi_seed_won\t{SEEDS[1]}\t{int((~won_by_first).sum())}")


## Coverage: how often each Measure is missing

A Measure is missing for a DayHike when it cannot be scored even against
itself (its fact is missing). It then drops out of every pair with that
DayHike and the other weights renormalize. Outside the sampled rasters the
climate, LandCover and SolarAspect facts are missing; Difficulty is missing
wherever no member way carries `sac_scale`.


In [ ]:
for measure in MEASURES:
    own = measure_scores(store, measure, everyone, everyone)
    print(f"measure_missing\t{measure.name}\t{np.isnan(own).mean():.3f}")


## Verification summary

Every `verify` line above (subject, check, score, PASS/FAIL), then the count
of passes and failures and each failure.


In [ ]:
summary = pd.DataFrame(
    [{"subject": subject, "check": check, "score": score, "result": result}
     for (subject, check), (score, result) in VERIFY.items()]
)
print(summary.round(4).to_string(index=False))
passes = int((summary["result"] == "PASS").sum())
print(f"verify_summary\tpass={passes}\tfail={len(summary) - passes}")
for row in summary[summary["result"] == "FAIL"].itertuples():
    print(f"verify_failed\t{row.subject}\t{row.check}\t{row.score:.3f}")
